# Tears of Steel: playback and scene analysis

This notebook makes *Tears of Steel* playable in your deployment and enables "ask about this scene" for it. It follows the v1 sample's [`bda-video.ipynb`](https://github.com/aws-samples/sample-hyper-personalized-agentic-ai-media-assistant/blob/v1/backend/tools/bda-video.ipynb), pointed at the resources this repository's stacks create.

*Tears of Steel* is (c) Blender Foundation | [mango.blender.org](https://mango.blender.org), licensed under [Creative Commons Attribution 3.0](https://creativecommons.org/licenses/by/3.0/). The film is not stored in this repository; this notebook downloads it from Blender's official mirrors.

What it does:

1. Downloads the 1080p film and converts it to a browser-playable MP4 (`tears-of-steel.mp4`).
2. Uploads it to your content bucket (`s3://<mediaBucket>/clips/movies/tears-of-steel.mp4`) and adds it to the video mapping table, so the title plays.
3. Analyses it with [Amazon Bedrock Data Automation](https://docs.aws.amazon.com/bedrock/latest/userguide/bda.html) (BDA): chapters with summaries, the transcript and the shots.
4. Optionally recognizes celebrities in one key frame per shot with Amazon Rekognition.
5. Writes the results to the `chapters`, `segments` and `audio-segments` indexes the scene-analysis agent reads.
6. Cuts two title-page highlights (action and romance, about 10 s each) that the app picks between by profile taste.

**Before you start**

- Deploy the stacks and run the data loaders (`infra/deployment/README.md`).
- Run this notebook with credentials for the deployment account: the deploying role, or a role listed in `dataLoaderPrincipals` in your environment file (the search collection only accepts writes from those).
- The role needs Bedrock Data Automation, S3 (content bucket), DynamoDB (video table) and, for step 4, `rekognition:RecognizeCelebrities`.
- About 2.5 GB of free disk space, and 15-25 minutes (the BDA job takes most of it). BDA bills per minute of video; the film is 12 minutes.

Re-running the notebook is safe: it replaces this video's documents rather than adding copies.

## Setup
Install the dependencies (`boto3` and an `ffmpeg` binary) and point the notebook at your environment file.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import json
from pathlib import Path
import tears_of_steel as tos

CONFIG = "../../infra/environments/<name>.json"   # your environment file
AWS_PROFILE = None                                 # or the name of a local AWS profile
LOCAL_FILM = None                                  # path to a film you already downloaded (.mov or .mp4), or None to download it
DETECT_CELEBRITIES = True                          # Rekognition on one frame per shot (a few hundred calls)
WORKDIR = Path("work")

dep = tos.Deployment(CONFIG, profile=AWS_PROFILE)

## 1. Get the film
Downloads the 1080p release (about 560 MB) from an official Blender mirror and checks its SHA-256. Skip the download by setting `LOCAL_FILM`.

In [ ]:
source = Path(LOCAL_FILM) if LOCAL_FILM else tos.download_film(WORKDIR)
film = tos.to_mp4(source, WORKDIR / tos.VIDEO_NAME)
print(film, f"{film.stat().st_size / 1e6:.0f} MB")

## 2. Make it playable
Upload the MP4 to the content bucket and add the title's row to the video mapping table. From then on, Play on the title page streams the film.

In [ ]:
dep.upload(film, dep.media_bucket, tos.VIDEO_KEY, content_type="video/mp4")
dep.seed_video_mapping()

## 3. Analyse the film with Bedrock Data Automation
The project uses the standard video output the v1 notebook enables: video and chapter summaries, IAB categories, the transcript, text in the frames and logos. The project is created once and reused on re-runs. BDA reads the film from, and writes its results to, the content bucket under `bda/tears-of-steel/`.

In [ ]:
project_arn = dep.bda_project(f"{dep.prefix}-tears-of-steel-video", {
    "video": {
        "extraction": {"category": {"state": "ENABLED", "types": ["TEXT_DETECTION", "TRANSCRIPT", "LOGOS"]},
                       "boundingBox": {"state": "ENABLED"}},
        "generativeField": {"state": "ENABLED", "types": ["VIDEO_SUMMARY", "CHAPTER_SUMMARY", "IAB"]},
    }
})
result = dep.run_bda(project_arn,
                     input_uri=f"s3://{dep.media_bucket}/{tos.VIDEO_KEY}",
                     output_uri=f"s3://{dep.media_bucket}/{tos.BDA_PREFIX}/video")
(WORKDIR / "bda-video-output.json").write_text(json.dumps(result))
print(len(result["chapters"]), "chapters,", len(result["shots"]), "shots")
print(result["video"]["summary"][:600])

## 4. Recognize celebrities (optional)
One key frame from the middle of each shot goes to Amazon Rekognition; names matched with at least 99% confidence are kept, as in v1. Tears of Steel's cast are mostly not in Rekognition's celebrity set, so expect few or no matches. The agent also looks up the cast from the catalog.

In [ ]:
celebrities = {}
if DETECT_CELEBRITIES:
    rekognition = dep.client("rekognition")
    for i, seconds in tos.shot_midpoints(result).items():
        faces = rekognition.recognize_celebrities(Image={"Bytes": tos.frame_jpeg(film, seconds)})["CelebrityFaces"]
        names = [c["Name"] for c in faces if c.get("MatchConfidence", 0) >= 99]
        if names:
            celebrities[i] = names
print(f"celebrities found in {len(celebrities)} shot(s):", sorted({n for v in celebrities.values() for n in v}))

## 5. Index the scenes
Write one document per chapter, per shot and per spoken segment. The scene-analysis agent finds them by `video_name` and the playback position.

In [ ]:
dep.replace_documents(tos.INDEXES["chapters"], tos.chapter_docs(result))
dep.replace_documents(tos.INDEXES["segments"], tos.segment_docs(result, celebrities))
dep.replace_documents(tos.INDEXES["audio"], tos.audio_docs(result))

## 6. Title-page highlights
Cut two highlights of about 10 seconds, an action scene and a romantic one, chosen from the chapter summaries above (the times are in `tears_of_steel.HIGHLIGHTS`). Each is cut in a horizontal and a vertical version and uploaded under `clips/highlights/vod/` with a one-sentence description in its S3 metadata (`x-amz-meta-description`, the same sentences as `infra/deployment/seed/highlight_descriptions.json`). When a profile opens the title page, the app plays the clip whose description is closest to that profile's taste.

In [ ]:
for variant, h in tos.HIGHLIGHTS.items():
    for orientation in ("horizontal", "vertical"):
        clip = tos.cut_highlight(film, h["start_s"], h["end_s"], orientation,
                                 WORKDIR / "highlights" / orientation / f"{tos.HIGHLIGHT_BASE}_{variant}.mp4")
        dep.upload(clip, dep.media_bucket, tos.highlight_key(variant, orientation),
                   content_type="video/mp4", metadata={"description": h["description"]})

## Done
Open the app, search for *Tears of Steel*, play it and ask about a scene, for example "What's happening in this scene?". New documents become searchable about a minute after indexing.

For answers grounded in the film's script, also run `2-script-knowledge-base.ipynb`.